# 06 - Model Training

Train XGBoost classifiers for each ticker to predict market direction from news features.


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report
from pathlib import Path

TICKERS = ['NVDA', 'AMD', 'AAPL', 'TSLA', 'MSFT']
MODELS_DIR = Path('../models/saved')

In [ ]:
FEATURE_COLS = [
    'polarity_score', 'subjectivity', 'compound_score',
    'severity_score', 'magnitude_estimate', 'rag_similarity',
    'is_earnings', 'is_merger', 'is_layoff', 'is_regulatory',
    'rsi_14', 'macd_signal', 'volume_ratio', 'price_momentum_5d'
]
TARGET_COL = 'direction_label'  # 0=Bearish, 1=Neutral, 2=Bullish

params = {
    'n_estimators': 200,
    'max_depth': 5,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'use_label_encoder': False,
    'eval_metric': 'mlogloss',
    'objective': 'multi:softprob',
    'num_class': 3,
    'random_state': 42
}

In [ ]:
for ticker in TICKERS:
    df = pd.read_csv(f'../data/processed/{ticker}_final_dataset.csv')
    X = df[FEATURE_COLS].fillna(0)
    y = df[TARGET_COL]
    
    model = xgb.XGBClassifier(**params)
    cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    print(f'{ticker} CV Accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')
    
    model.fit(X, y)
    model.save_model(MODELS_DIR / f'xgb_{ticker}_latest.json')
    print(f'Saved {ticker} model')